<a href="https://colab.research.google.com/github/Renad-Alzahrani-gif/Classification-model/blob/main/Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
upload=files.upload()

Saving train (1).csv to train (1).csv


In [ ]:
import pandas as pd
read=pd.read_csv("train (1).csv")
read.head()
read.shape
read.info()

print(read["Survived"].value_counts())
print(read["Survived"].value_counts(normalize=True))
print(read.describe())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
Survived
0    549
1    342
Name: count, dtype: int64
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier


y = read["Survived"]

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = read[features]


X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, random_state=0
)

categorical_cols = X_train.select_dtypes(include=["object"]).columns
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns

print("Categorical:")
print(categorical_cols)

print("\nNumerical:")
print(numerical_cols)

#Clean Data
numerical_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


# Model
model=LogisticRegression(max_iter=1000)

mypipline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

mypipline.fit(X_train,y_train)


predictions = mypipline.predict(X_valid)


#evaluation
accuracy = accuracy_score(y_valid, predictions)

print("Accuracy:", accuracy)

cm = confusion_matrix(y_valid, predictions)

print(cm)

print(classification_report(y_valid, predictions))


Categorical:
Index(['Sex', 'Embarked'], dtype='object')

Numerical:
Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')
Accuracy: 0.7937219730941704
[[116  23]
 [ 23  61]]
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       139
           1       0.73      0.73      0.73        84

    accuracy                           0.79       223
   macro avg       0.78      0.78      0.78       223
weighted avg       0.79      0.79      0.79       223



In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score



# 1. Data


read = pd.read_csv("train (1).csv")

y = read["Survived"]

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = read[features]



# 2. Train / Validation Split


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    random_state=0
)


# 3. Identify Columns

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns

numerical_cols = X_train.select_dtypes(
    exclude=["object"]
).columns



# 4. Preprocessing


numerical_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)



# 5. Logistic Regression


model = LogisticRegression(max_iter=1000)

mypipline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

mypipline.fit(X_train, y_train)

predictions = mypipline.predict(X_valid)



# 6. Logistic Regression Evaluation


accuracy = accuracy_score(y_valid, predictions)

print("Logistic Regression")
print("Accuracy:", accuracy)

cm = confusion_matrix(y_valid, predictions)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_valid, predictions))

cv_scores_lr = cross_val_score(
    mypipline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Logistic Regression CV scores:")
print(cv_scores_lr)

print("Mean:", cv_scores_lr.mean())



# 7. Random Forest Classifier

model_rf = RandomForestClassifier(
    n_estimators=100,
        max_depth=10,
    random_state=0
)

mypipline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model_rf)
])

mypipline_rf.fit(X_train, y_train)

predictions_rf = mypipline_rf.predict(X_valid)



# 8. Random Forest Evaluation


accuracy_rf = accuracy_score(y_valid, predictions_rf)

print("\n Random Forest ")
print("Accuracy:", accuracy_rf)

cm_rf = confusion_matrix(y_valid, predictions_rf)

print("\nConfusion Matrix:")
print(cm_rf)

print("\nClassification Report:")
print(classification_report(y_valid, predictions_rf))
cv_scores_rf = cross_val_score(
    mypipline_rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Random Forest CV scores:")
print(cv_scores_rf)

print("Mean:", cv_scores_rf.mean())



===== Logistic Regression =====
Accuracy: 0.7937219730941704

Confusion Matrix:
[[116  23]
 [ 23  61]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       139
           1       0.73      0.73      0.73        84

    accuracy                           0.79       223
   macro avg       0.78      0.78      0.78       223
weighted avg       0.79      0.79      0.79       223

Logistic Regression CV scores:
[0.77094972 0.78651685 0.78089888 0.76966292 0.82022472]
Mean: 0.7856506182913816

===== Random Forest =====
Accuracy: 0.8385650224215246

Confusion Matrix:
[[127  12]
 [ 24  60]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.91      0.88       139
           1       0.83      0.71      0.77        84

    accuracy                           0.84       223
   macro avg       0.84      0.81      0.82       223
weighted avg       0.84      0.84      

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# 1. Data


read = pd.read_csv("train (1).csv")

#Features Engineering

read["Title"] = read["Name"].str.extract(r",\s*([^.]*)\.")
read["FamilySize"] = read["SibSp"] + read["Parch"] + 1
read["IsAlone"] = (read["FamilySize"] == 1).astype(int)

y = read["Survived"]

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Title",
    "FamilySize",
    "IsAlone",


]

X = read[features]



# 2. Train / Validation Split


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    random_state=0

)

# 3. Identify Columns

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns

numerical_cols = X_train.select_dtypes(
    exclude=["object"]
).columns


# 4. Preprocessing


numerical_transformer = SimpleImputer(
    strategy="median"
)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# 5. Logistic Regression


model = LogisticRegression(max_iter=1000)

mypipline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

mypipline.fit(X_train, y_train)

predictions = mypipline.predict(X_valid)

# 6. Logistic Regression Evaluation


accuracy = accuracy_score(y_valid, predictions)

print("===== Logistic Regression =====")
print("Accuracy:", accuracy)

cm = confusion_matrix(y_valid, predictions)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_valid, predictions))


# 7. Random Forest


model_rf = RandomForestClassifier(
    random_state=0

)

mypipline_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model_rf)
])

# 8. GridSearchCV


param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [5, 7, 10],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", None]
}

grid_search = GridSearchCV(
    mypipline_rf,
    param_grid,
    cv=5,
scoring="accuracy"
)


grid_search.fit(X_train, y_train)


# 9. Best Parameters


print("\n Grid Search ")

print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest CV score:")
print(grid_search.best_score_)


# 10. Best Model

best_model = grid_search.best_estimator_

#    Validation
predictions_best = best_model.predict(X_valid)


# 11. Best Model Evaluation


accuracy_best = accuracy_score(
    y_valid,
    predictions_best
)

print("\n Best Random Forest ")
print("Validation Accuracy:", accuracy_best)

cm_best = confusion_matrix(
    y_valid,
    predictions_best
)

print("\nConfusion Matrix:")
print(cm_best)

print("\nClassification Report:")
print(
    classification_report(
        y_valid,
        predictions_best
    )
)

===== Logistic Regression =====
Accuracy: 0.8116591928251121

Confusion Matrix:
[[115  24]
 [ 18  66]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.85       139
           1       0.73      0.79      0.76        84

    accuracy                           0.81       223
   macro avg       0.80      0.81      0.80       223
weighted avg       0.82      0.81      0.81       223


===== Grid Search =====

Best parameters:
{'classifier__max_depth': 5, 'classifier__max_features': None, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 200}

Best CV score:
0.8473347547974412

===== Best Random Forest =====
Validation Accuracy: 0.8251121076233184

Confusion Matrix:
[[120  19]
 [ 20  64]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       139
           1       0.77      0.76      0.77        84

    accuracy                     